# Calculate climatologies for the NPP observational products

In [1]:
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

In [2]:
basedir_out = "/projects/SOCCOM/graemem/projects/esm4_ppe/data/processed/observations/"
localdir_obs = "NPP-RyanKeogh/NPP_MLD-EN4.2.2-0.03/"

In [3]:
# Load observational data
basedir_obs = "/projects/SOCCOM/datasets/"
products = ["CAFE","CbPM1","CbPM2","EPPLEY","VGPM"]
suffix_obs = "_NPP_8D_25KM_1998_2022.nc"
ds_obs = xr.Dataset()
for product in products:
    filename = product+suffix_obs
    ds_obs[product] = xr.open_dataarray(basedir_obs+localdir_obs+filename)

In [14]:
ds_obs = ds_obs.chunk({'time':-1,'lat':'auto','lon':'auto'})

## Time mean

In [15]:
with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ds_obs_climmean = ds_obs.groupby('time.month').mean()

In [16]:
with ProgressBar():
    ds_obs_climmean = ds_obs_climmean.compute()

[########################################] | 100% Completed | 511.29 s


In [18]:
filename = "AllProducts_8D_25KM_1998_2022_climmean.nc"
ds_obs_climmean.to_netcdf(basedir_out+localdir_obs+filename)

## Anomaly variance

In [20]:
# Reload climatological mean
ds_obs_climmean = xr.open_dataset(basedir_out+localdir+filename)

In [29]:
with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ds_obs_climanomvar = (ds_obs.groupby('time.month') - ds_obs_climmean).groupby('time.month').std()**2

/home/graemem/.conda/envs/core/lib/python3.10/site-packages/xarray/core/indexing.py:1446: PerformanceWarning: Slicing with an out-of-order index is generating 25 times more chunks
  return self.array[key]


In [31]:
with ProgressBar():
    ds_obs_climanomvar = ds_obs_climanomvar.compute()

[                                        ] | 0% Completed | 40.57 sms

/home/graemem/.conda/envs/core/lib/python3.10/site-packages/dask/array/numpy_compat.py:40: RuntimeWarning: invalid value encountered in true_divide
  x = np.divide(x1, x2, out)


[########################################] | 100% Completed | 14m 27s


In [32]:
filename = "AllProducts_8D_25KM_1998_2022_climanomvar.nc"
ds_obs_climmean.to_netcdf(basedir_out+localdir+filename)

In [4]:
ds_obs

<xarray.Dataset>
Dimensions:  (lon: 1440, lat: 720, time: 1150)
Coordinates:
  * lon      (lon) float32 -179.9 -179.6 -179.4 -179.1 ... 179.4 179.6 179.9
  * lat      (lat) float32 -89.88 -89.62 -89.38 -89.12 ... 89.38 89.62 89.88
  * time     (time) datetime64[ns] 1998-01-01 1998-01-09 ... 2022-12-27
Data variables:
    CAFE     (time, lat, lon) float64 ...
    CbPM1    (time, lat, lon) float64 ...
    CbPM2    (time, lat, lon) float64 ...
    EPPLEY   (time, lat, lon) float64 ...
    VGPM     (time, lat, lon) float64 ...